# 25 — Silver Pilot Preflight

## Objetivo

Convertir la especificación del notebook 24 en un plan operativo de piloto antes de recolectar datos reales.

El piloto se fija, como criterio operativo del proyecto, en:

- 12 sitios educativos,
- 2 tareas por sitio,
- 2 ejecuciones por tarea,
- 48 instancias totales.

Este notebook:

1. crea plantillas concretas para 12 sitios y 24 tareas;
2. valida que los registros estén completos antes de recolectar;
3. construye el `pilot_manifest.csv`;
4. valida exportaciones reales de la extensión cuando aparezcan;
5. comprueba unicidad, columnas X, rangos y faltantes;
6. prepara paquetes de anotación Silver;
7. genera un estado `READY / NOT_READY`.

No entrena modelos y no modifica `data/processed/`.


## 1. Configuración

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 400)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"

SILVER_DIR = PROJECT_DIR / "data" / "silver"
META_DIR = SILVER_DIR / "metadata"
RAW_DIR = SILVER_DIR / "raw"
LABEL_DIR = SILVER_DIR / "labels"
INTERIM_DIR = SILVER_DIR / "interim"

EXPORT_DIR = RAW_DIR / "extension_exports"
EVIDENCE_DIR = RAW_DIR / "evidence"
ANNOTATION_PACKAGE_DIR = INTERIM_DIR / "annotation_packages"

for directory in [
    REPORTS_DIR,
    SILVER_DIR,
    META_DIR,
    RAW_DIR,
    LABEL_DIR,
    INTERIM_DIR,
    EXPORT_DIR,
    EVIDENCE_DIR,
    ANNOTATION_PACKAGE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("PROJECT_DIR:", PROJECT_DIR)
print("SILVER_DIR:", SILVER_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning
SILVER_DIR: /Users/uriel/Documents/machine_learning/data/silver


## 2. Cargar contratos del notebook 24

In [21]:
capture_contract = pd.read_csv(
    REPORTS_DIR
    / "silver_extension_capture_contract.csv"
)

target_rubric = pd.read_csv(
    REPORTS_DIR
    / "silver_target_rubric.csv"
)

collection_targets = pd.read_csv(
    REPORTS_DIR
    / "silver_collection_targets.csv"
)

split_policy = pd.read_csv(
    REPORTS_DIR
    / "silver_split_policy.csv"
)

print("Contrato captura:", capture_contract.shape)
print("Target rubric:", target_rubric.shape)
print("Collection targets:", collection_targets.shape)

X_NAMES = (
    capture_contract.loc[
        capture_contract["role"] == "X",
        "column"
    ]
    .astype(str)
    .tolist()
)

print("X canónicas:", len(X_NAMES))

assert len(X_NAMES) == 40


Contrato captura: (46, 4)
Target rubric: (4, 5)
Collection targets: (2, 5)
X canónicas: 40


## 3. Diseño operativo del piloto

Para obtener 48 instancias sin tratar ejecuciones repetidas como sitios independientes:

\[
12\ 	ext{sitios}
	imes 2\ 	ext{tareas/sitio}
	imes 2\ 	ext{ejecuciones/tarea}
= 48\ 	ext{instancias}
\]

Las repeticiones sirven para comprobar estabilidad de captura, no para inflar artificialmente el número de sitios.


In [22]:
PILOT_SITES = 12
TASKS_PER_SITE = 2
EXECUTIONS_PER_TASK = 2

EXPECTED_TASKS = (
    PILOT_SITES
    *
    TASKS_PER_SITE
)

EXPECTED_INSTANCES = (
    EXPECTED_TASKS
    *
    EXECUTIONS_PER_TASK
)

print("Sitios:", PILOT_SITES)
print("Tareas:", EXPECTED_TASKS)
print("Instancias:", EXPECTED_INSTANCES)

assert EXPECTED_INSTANCES == 48


Sitios: 12
Tareas: 24
Instancias: 48


## 4. Crear plantilla de 12 sitios

In [23]:
site_template_path = (
    META_DIR
    / "site_registry_pilot.csv"
)

site_columns = [
    "site_id",
    "site_name",
    "base_url",
    "domain",
    "educational_context",
    "access_type",
    "requires_account",
    "uses_personal_data",
    "site_version_or_date",
    "included",
    "exclusion_reason",
    "notes"
]

if not site_template_path.exists():

    site_seed = pd.DataFrame([
        {
            "site_id":
                f"site_{i:03d}",

            "site_name":
                "",

            "base_url":
                "",

            "domain":
                "",

            "educational_context":
                "",

            "access_type":
                "",

            "requires_account":
                False,

            "uses_personal_data":
                False,

            "site_version_or_date":
                "",

            "included":
                False,

            "exclusion_reason":
                "",

            "notes":
                ""
        }
        for i in range(
            1,
            PILOT_SITES + 1
        )
    ])

    site_seed.to_csv(
        site_template_path,
        index=False
    )

site_registry = pd.read_csv(
    site_template_path
)

print(
    "Site registry:",
    site_registry.shape
)

display(
    site_registry
)


Site registry: (12, 12)


,site_id,site_name,base_url,domain,educational_context,access_type,requires_account,uses_personal_data,site_version_or_date,included,exclusion_reason,notes
0,site_001,PhET Interactive Simulations,https://phet.colorado.edu/,phet.colorado.edu,Simulaciones interactivas de ciencias y matemáticas,PUBLIC,False,False,2026-09-07,True,NaN,Sitio público. Verificar disponibilidad del simulador seleccionado inmediatamente antes de la captura.
1,site_002,GeoGebra Calculator Suite,https://www.geogebra.org/calculator,geogebra.org,"Matemáticas dinámicas: álgebra, geometría, cálculo y gráficas",PUBLIC,False,False,2026-09-07,True,NaN,La calculadora web puede utilizarse sin guardar contenido. No iniciar sesión durante el piloto.
2,site_003,Desmos Graphing Calculator,https://www.desmos.com/calculator,desmos.com,Calculadora gráfica interactiva,PUBLIC,False,False,2026-09-07,True,NaN,Utilizar únicamente funciones locales de la calculadora; no guardar ni compartir.
3,site_004,Scratch Online Editor,https://scratch.mit.edu/projects/editor/,scratch.mit.edu,Programación visual educativa,PUBLIC,False,False,2026-09-07,True,NaN,Usar un proyecto temporal sin iniciar sesión ni publicar.
4,site_005,Polypad,https://mathigon.org/polypad,mathigon.org,Manipulativos virtuales para matemáticas,PUBLIC,False,False,2026-09-07,True,NaN,Actualmente redirige al entorno Polypad de Amplify. No guardar ni compartir actividades.
5,site_006,Numbas,https://numbas.mathcentre.ac.uk/,numbas.mathcentre.ac.uk,Sistema de evaluación matemática web,PUBLIC,False,False,2026-09-07,True,NaN,Usar exclusivamente la base de datos pública y previews de exámenes; no crear cuenta.
6,site_007,H5P Examples,https://h5p.org/content-types-and-applications,h5p.org,Ejemplos de contenido educativo interactivo H5P,PUBLIC,False,False,2026-09-07,True,NaN,Usar previews públicos de tipos de contenido; no crear contenido ni cuenta.
7,site_008,MDN Playground,https://developer.mozilla.org/en-US/play,developer.mozilla.org,"Aprendizaje y experimentación con HTML, CSS y JavaScript",PUBLIC,False,False,2026-09-07,True,NaN,Trabajar con código mínimo local en el playground; no iniciar sesión.
8,site_009,W3Schools Tryit Editor,https://www.w3schools.com/tryit/,w3schools.com,Editor y compiladores educativos en navegador,PUBLIC,False,False,2026-09-07,True,NaN,El inicio de sesión para seguimiento no es necesario para las tareas propuestas.
9,site_010,freeCodeCamp Learn,https://www.freecodecamp.org/learn,freecodecamp.org,Currículo interactivo de programación web,PUBLIC,False,False,2026-09-07,True,NaN,"Usar lecciones públicas sin crear cuenta. Si una tarea exige login, marcar el sitio/tarea como excluido y sustituirlo."


### Qué llenar en `site_registry_pilot.csv`

Para cada uno de los 12 sitios:

- `site_name`: nombre visible del sitio;
- `base_url`: URL principal;
- `domain`: dominio;
- `educational_context`: curso, LMS, simulador, práctica, biblioteca educativa, etc.;
- `access_type`: `PUBLIC`, `DEMO` o `AUTHORIZED_TEST_ACCOUNT`;
- `requires_account`: `True/False`;
- `uses_personal_data`: debe ser `False` para este piloto;
- `included`: `True` solo cuando el sitio haya sido revisado y aceptado.

No usar cuentas ajenas, datos personales reales ni flujos de pago.


## 5. Crear plantilla de 24 tareas

In [24]:
task_template_path = (
    META_DIR
    / "task_registry_pilot.csv"
)

task_columns = [
    "task_id",
    "site_id",
    "task_name",
    "task_description",
    "start_condition",
    "success_criterion",
    "failure_criterion",
    "expected_page_type",
    "expected_interaction_type",
    "applicable_H1",
    "applicable_H4",
    "applicable_H5",
    "applicable_H6",
    "included",
    "notes"
]

if not task_template_path.exists():

    task_seed_rows = []

    for site_index in range(
        1,
        PILOT_SITES + 1
    ):

        site_id = (
            f"site_{site_index:03d}"
        )

        for task_index in range(
            1,
            TASKS_PER_SITE + 1
        ):

            task_seed_rows.append({
                "task_id":
                    (
                        f"{site_id}"
                        f"__task_{task_index:02d}"
                    ),

                "site_id":
                    site_id,

                "task_name":
                    "",

                "task_description":
                    "",

                "start_condition":
                    "",

                "success_criterion":
                    "",

                "failure_criterion":
                    "",

                "expected_page_type":
                    "",

                "expected_interaction_type":
                    "",

                "applicable_H1":
                    np.nan,

                "applicable_H4":
                    np.nan,

                "applicable_H5":
                    np.nan,

                "applicable_H6":
                    np.nan,

                "included":
                    False,

                "notes":
                    ""
            })

    pd.DataFrame(
        task_seed_rows
    ).to_csv(
        task_template_path,
        index=False
    )

task_registry = pd.read_csv(
    task_template_path
)

print(
    "Task registry:",
    task_registry.shape
)

display(
    task_registry
)


Task registry: (24, 15)


,task_id,site_id,task_name,task_description,start_condition,success_criterion,failure_criterion,expected_page_type,expected_interaction_type,applicable_H1,applicable_H4,applicable_H5,applicable_H6,included,notes
0,site_001__task_01,site_001,Abrir una simulación de física,"Desde la página principal, entrar a Física y abrir una simulación interactiva disponible.",Página principal de PhET cargada.,La simulación elegida queda abierta y lista para interactuar.,No se logra abrir ninguna simulación pública.,simulation_catalog_to_simulation,navigation,True,True,False,True,True,Usar la misma simulación en ambas ejecuciones.
1,site_001__task_02,site_001,Modificar un parámetro de simulación,"Dentro de la simulación seleccionada, cambiar un control visible y observar la respuesta del sistema.",Simulación cargada en su estado inicial.,Se modifica un parámetro y se observa un cambio visible en la simulación.,La simulación no responde o el control no puede utilizarse.,interactive_simulation,control_manipulation,True,True,False,True,True,Registrar exactamente qué control se usa.
2,site_002__task_01,site_002,Graficar y=x^2,Introducir la expresión y=x^2 en la Suite Calculadora y visualizar la gráfica.,GeoGebra Calculator Suite abierta sin expresiones previas.,La función y=x^2 aparece representada en la vista gráfica.,La expresión no puede introducirse o la gráfica no aparece.,graphing_calculator,text_input_and_render,True,True,True,True,True,NaN
3,site_002__task_02,site_002,Cambiar de vista y volver,"Con la función ya creada, abrir una vista/herramienta secundaria disponible y regresar a la vista gráfica.",Función y=x^2 visible.,Se cambia de vista y se regresa conservando la expresión o estado esperado.,No se puede cambiar de vista o se pierde el estado inesperadamente.,multi_view_math_app,navigation_with_state,True,True,False,True,True,NaN
4,site_003__task_01,site_003,Graficar y=x^2,Introducir y=x^2 y verificar que aparezca la curva.,Calculadora Desmos abierta y limpia.,La curva y=x^2 aparece en el plano.,La expresión no se acepta o no se visualiza.,graphing_calculator,text_input_and_render,True,True,True,True,True,NaN
5,site_003__task_02,site_003,Crear y modificar un deslizador,"Introducir y=a*x, crear el deslizador de a y modificar su valor.",Calculadora limpia o con una expresión que pueda eliminarse.,El deslizador existe y el cambio de a modifica visualmente la gráfica.,No se crea el deslizador o el cambio no produce feedback visible.,graphing_calculator,parameter_manipulation,True,True,True,True,True,NaN
6,site_004__task_01,site_004,Agregar y ejecutar un bloque de movimiento,Añadir al sprite un bloque 'mover 10 pasos' y ejecutarlo mediante interacción directa.,Editor Scratch abierto con proyecto temporal nuevo.,El sprite cambia de posición tras ejecutar el bloque.,No se puede agregar o ejecutar el bloque.,visual_programming_editor,drag_drop_and_execute,True,True,False,True,True,No guardar ni publicar.
7,site_004__task_02,site_004,Crear secuencia de dos bloques,Agregar dos bloques compatibles y ejecutar la secuencia observando el resultado.,Proyecto temporal con sprite disponible.,Los dos bloques quedan conectados y la secuencia produce un resultado visible.,Los bloques no pueden conectarse o no se observa resultado.,visual_programming_editor,drag_drop_sequence,True,True,True,True,True,NaN
8,site_005__task_01,site_005,Agregar manipulativos matemáticos,Crear un Polypad nuevo y añadir al menos dos objetos matemáticos visibles.,Polypad abierto en un tablero nuevo.,Dos objetos quedan colocados en el tablero.,No se pueden añadir objetos.,virtual_manipulative_canvas,drag_drop,True,True,False,True,True,NaN
9,site_005__task_02,site_005,Modificar y eliminar un objeto,"Seleccionar uno de los objetos, modificar una propiedad disponible y después eliminarlo.",Tablero con al menos dos objetos.,La modificación se refleja y el objeto puede eliminarse.,No se identifica el objeto o no es posible modificar/eliminar.,virtual_manipulative_canvas,object_editin

### Qué llenar en `task_registry_pilot.csv`

Cada tarea debe:

- tener un inicio reproducible;
- tener criterio de éxito observable;
- poder repetirse dos veces;
- no requerir datos personales;
- permitir al menos una oportunidad razonable para evaluar H1/H4/H5/H6;
- mantener `NaN` en `applicable_H*` cuando aún no se haya determinado la aplicabilidad.

`applicable_H*` no es la etiqueta.


## 6. Validar registro de sitios

In [25]:
def normalize_bool_series(
    series
):

    mapping = {
        True: True,
        False: False,
        "true": True,
        "false": False,
        "True": True,
        "False": False,
        "1": True,
        "0": False,
        1: True,
        0: False
    }

    return series.map(
        mapping
    )


site_registry = pd.read_csv(
    site_template_path
)

for col in [
    "included",
    "requires_account",
    "uses_personal_data"
]:
    site_registry[
        col
    ] = normalize_bool_series(
        site_registry[
            col
        ]
    )

included_sites = site_registry[
    site_registry[
        "included"
    ]
    ==
    True
].copy()

site_checks = []

site_checks.append({
    "check":
        "TOTAL_INCLUDED_SITES",

    "passed":
        len(
            included_sites
        )
        ==
        PILOT_SITES,

    "observed":
        len(
            included_sites
        ),

    "expected":
        PILOT_SITES
})

site_checks.append({
    "check":
        "UNIQUE_SITE_ID",

    "passed":
        included_sites[
            "site_id"
        ].duplicated().sum()
        ==
        0,

    "observed":
        int(
            included_sites[
                "site_id"
            ].duplicated().sum()
        ),

    "expected":
        0
})

required_site_text = [
    "site_name",
    "base_url",
    "domain",
    "educational_context",
    "access_type"
]

for col in required_site_text:

    missing = (
        included_sites[
            col
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        ==
        ""
    ).sum()

    site_checks.append({
        "check":
            f"SITE_REQUIRED_{col}",

        "passed":
            int(
                missing
            )
            ==
            0,

        "observed":
            int(
                missing
            ),

        "expected":
            0
    })

personal_data_true = (
    included_sites[
        "uses_personal_data"
    ]
    ==
    True
).sum()

site_checks.append({
    "check":
        "NO_PERSONAL_DATA",

    "passed":
        int(
            personal_data_true
        )
        ==
        0,

    "observed":
        int(
            personal_data_true
        ),

    "expected":
        0
})

site_validation = pd.DataFrame(
    site_checks
)

display(
    site_validation
)


,check,passed,observed,expected
0,TOTAL_INCLUDED_SITES,True,12,12
1,UNIQUE_SITE_ID,True,0,0
2,SITE_REQUIRED_site_name,True,0,0
3,SITE_REQUIRED_base_url,True,0,0
4,SITE_REQUIRED_domain,True,0,0
5,SITE_REQUIRED_educational_context,True,0,0
6,SITE_REQUIRED_access_type,True,0,0
7,NO_PERSONAL_DATA,True,0,0


## 7. Validar registro de tareas

In [26]:
task_registry = pd.read_csv(
    task_template_path
)

task_registry[
    "included"
] = normalize_bool_series(
    task_registry[
        "included"
    ]
)

included_tasks = task_registry[
    task_registry[
        "included"
    ]
    ==
    True
].copy()

task_checks = []

task_checks.append({
    "check":
        "TOTAL_INCLUDED_TASKS",

    "passed":
        len(
            included_tasks
        )
        ==
        EXPECTED_TASKS,

    "observed":
        len(
            included_tasks
        ),

    "expected":
        EXPECTED_TASKS
})

task_checks.append({
    "check":
        "UNIQUE_TASK_ID",

    "passed":
        included_tasks[
            "task_id"
        ].duplicated().sum()
        ==
        0,

    "observed":
        int(
            included_tasks[
                "task_id"
            ].duplicated().sum()
        ),

    "expected":
        0
})

unknown_sites = (
    set(
        included_tasks[
            "site_id"
        ]
    )
    -
    set(
        included_sites[
            "site_id"
        ]
    )
)

task_checks.append({
    "check":
        "TASK_SITE_FOREIGN_KEY",

    "passed":
        len(
            unknown_sites
        )
        ==
        0,

    "observed":
        str(
            sorted(
                unknown_sites
            )
        ),

    "expected":
        "[]"
})

required_task_text = [
    "task_name",
    "task_description",
    "start_condition",
    "success_criterion"
]

for col in required_task_text:

    missing = (
        included_tasks[
            col
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        ==
        ""
    ).sum()

    task_checks.append({
        "check":
            f"TASK_REQUIRED_{col}",

        "passed":
            int(
                missing
            )
            ==
            0,

        "observed":
            int(
                missing
            ),

        "expected":
            0
    })

tasks_per_site = (
    included_tasks
    .groupby(
        "site_id"
    )
    .size()
)

bad_task_counts = (
    tasks_per_site[
        tasks_per_site
        !=
        TASKS_PER_SITE
    ]
)

task_checks.append({
    "check":
        "TASKS_PER_SITE",

    "passed":
        len(
            bad_task_counts
        )
        ==
        0
        and
        len(
            tasks_per_site
        )
        ==
        PILOT_SITES,

    "observed":
        str(
            tasks_per_site
            .to_dict()
        ),

    "expected":
        (
            f"{TASKS_PER_SITE} por sitio"
        )
})

task_validation = pd.DataFrame(
    task_checks
)

display(
    task_validation
)


,check,passed,observed,expected
0,TOTAL_INCLUDED_TASKS,True,24,24
1,UNIQUE_TASK_ID,True,0,0
2,TASK_SITE_FOREIGN_KEY,True,[],[]
3,TASK_REQUIRED_task_name,True,0,0
4,TASK_REQUIRED_task_description,True,0,0
5,TASK_REQUIRED_start_condition,True,0,0
6,TASK_REQUIRED_success_criterion,True,0,0
7,TASKS_PER_SITE,True,"{'site_001': 2, 'site_002': 2, 'site_003': 2, 'site_004': 2, 'site_005': 2, 'site_006': 2, 'site_007': 2, 'site_008': 2, 'site_009': 2, 'site_010': 2, 'site_011': 2, 'site_012': 2}",2 por sitio


## 8. Construir manifiesto de 48 instancias

In [27]:
registries_ready = (
    site_validation[
        "passed"
    ].all()
    and
    task_validation[
        "passed"
    ].all()
)

pilot_manifest_path = (
    META_DIR
    / "pilot_manifest.csv"
)

if registries_ready:

    manifest_rows = []

    for _, task in (
        included_tasks.iterrows()
    ):

        for execution_index in range(
            1,
            EXECUTIONS_PER_TASK + 1
        ):

            execution_id = (
                f"exec_{execution_index:02d}"
            )

            instance_id = (
                f"{task['site_id']}"
                f"__{task['task_id'].split('__')[-1]}"
                f"__{execution_id}"
            )

            manifest_rows.append({
                "instance_id":
                    instance_id,

                "site_id":
                    task[
                        "site_id"
                    ],

                "task_id":
                    task[
                        "task_id"
                    ],

                "execution_id":
                    execution_id,

                "collection_status":
                    "PLANNED",

                "extension_export_path":
                    (
                        "data/silver/raw/"
                        "extension_exports/"
                        f"{instance_id}.csv"
                    ),

                "evidence_dir":
                    (
                        "data/silver/raw/"
                        "evidence/"
                        f"{instance_id}"
                    ),

                "label_runs_required":
                    3
            })

    pilot_manifest = pd.DataFrame(
        manifest_rows
    )

    assert len(
        pilot_manifest
    ) == EXPECTED_INSTANCES

    assert (
        pilot_manifest[
            "instance_id"
        ]
        .duplicated()
        .sum()
        ==
        0
    )

    pilot_manifest.to_csv(
        pilot_manifest_path,
        index=False
    )

else:

    pilot_manifest = pd.DataFrame(
        columns=[
            "instance_id",
            "site_id",
            "task_id",
            "execution_id",
            "collection_status",
            "extension_export_path",
            "evidence_dir",
            "label_runs_required"
        ]
    )

    pilot_manifest.to_csv(
        pilot_manifest_path,
        index=False
    )

print(
    "Registries ready:",
    registries_ready
)

print(
    "Pilot manifest:",
    pilot_manifest.shape
)

display(
    pilot_manifest.head(
        20
    )
)


Registries ready: True
Pilot manifest: (48, 8)


,instance_id,site_id,task_id,execution_id,collection_status,extension_export_path,evidence_dir,label_runs_required
0,site_001__task_01__exec_01,site_001,site_001__task_01,exec_01,PLANNED,data/silver/raw/extension_exports/site_001__task_01__exec_01.csv,data/silver/raw/evidence/site_001__task_01__exec_01,3
1,site_001__task_01__exec_02,site_001,site_001__task_01,exec_02,PLANNED,data/silver/raw/extension_exports/site_001__task_01__exec_02.csv,data/silver/raw/evidence/site_001__task_01__exec_02,3
2,site_001__task_02__exec_01,site_001,site_001__task_02,exec_01,PLANNED,data/silver/raw/extension_exports/site_001__task_02__exec_01.csv,data/silver/raw/evidence/site_001__task_02__exec_01,3
3,site_001__task_02__exec_02,site_001,site_001__task_02,exec_02,PLANNED,data/silver/raw/extension_exports/site_001__task_02__exec_02.csv,data/silver/raw/evidence/site_001__task_02__exec_02,3
4,site_002__task_01__exec_01,site_002,site_002__task_01,exec_01,PLANNED,data/silver/raw/extension_exports/site_002__task_01__exec_01.csv,data/silver/raw/evidence/site_002__task_01__exec_01,3
5,site_002__task_01__exec_02,site_002,site_002__task_01,exec_02,PLANNED,data/silver/raw/extension_exports/site_002__task_01__exec_02.csv,data/silver/raw/evidence/site_002__task_01__exec_02,3
6,site_002__task_02__exec_01,site_002,site_002__task_02,exec_01,PLANNED,data/silver/raw/extension_exports/site_002__task_02__exec_01.csv,data/silver/raw/evidence/site_002__task_02__exec_01,3
7,site_002__task_02__exec_02,site_002,site_002__task_02,exec_02,PLANNED,data/silver/raw/extension_exports/site_002__task_02__exec_02.csv,data/silver/raw/evidence/site_002__task_02__exec_02,3
8,site_003__task_01__exec_01,site_003,site_003__task_01,exec_01,PLANNED,data/silver/raw/extension_exports/site_003__task_01__exec_01.csv,data/silver/raw/evidence/site_003__task_01__exec_01,3
9,site_003__task_01__exec_02,site_003,site_003__task_01,exec_02,PLANNED,data/silver/raw/extension_exports/site_003__task_01__exec_02.csv,data/silver/raw/evidence/site_003__task_01__exec_02,3


Si `Registries ready=False`, el notebook no está fallando: significa que primero deben completarse `site_registry_pilot.csv` y `task_registry_pilot.csv`.


## 9. Contrato mínimo de cada exportación de la extensión

In [28]:
CONTEXT_COLUMNS = [
    "instance_id",
    "site_id",
    "task_id",
    "execution_id",
    "platform",
    "page_type"
]

REQUIRED_EXPORT_COLUMNS = (
    CONTEXT_COLUMNS
    +
    X_NAMES
)

print(
    "Columnas esperadas por export:",
    len(
        REQUIRED_EXPORT_COLUMNS
    )
)

assert len(
    REQUIRED_EXPORT_COLUMNS
) == 46


Columnas esperadas por export: 46


## 10. Reglas de rango para X

In [29]:
RATIO_COLUMNS = [
    "feedback_coverage_ratio",
    "component_style_consistency_ratio",
    "css_class_consistency_ratio",
    "navigation_consistency_ratio",
    "label_consistency_ratio",
    "semantic_role_consistency_ratio",
    "dom_structure_similarity",
    "navigation_persistence_ratio",
    "accessible_name_ratio",
    "visible_labels_ratio",
    "visible_options_ratio"
]

BINARY_COLUMNS = [
    "feedback_present",
    "loader_present",
    "progress_indicator_present",
    "status_message_present",
    "completion_feedback_present",
    "input_validation_present",
    "client_validation_present",
    "invalid_submission_blocked",
    "confirmation_dialog_present",
    "destructive_action_protected",
    "error_feedback_present",
    "breadcrumbs_present"
]

COUNT_COLUMNS = [
    "click_count",
    "keypress_count",
    "scroll_count",
    "backtracking_count",
    "repeated_action_count",
    "navigation_steps",
    "validation_error_count",
    "user_error_count",
    "unlabeled_controls_count"
]

NONNEGATIVE_COLUMNS = [
    "task_time_s",
    "mean_response_time_ms",
    "max_response_time_ms",
    "mean_pre_action_pause_s",
    "feedback_latency_ms",
    "loader_duration_ms",
    "navigation_depth"
]

for col in (
    RATIO_COLUMNS
    +
    BINARY_COLUMNS
    +
    COUNT_COLUMNS
    +
    NONNEGATIVE_COLUMNS
):
    if col not in X_NAMES:
        print(
            "Advertencia:",
            col,
            "no está en X_NAMES"
        )


## 11. Función de validación de export

In [30]:
def validate_extension_export(
    path
):

    result = {
        "file":
            str(
                path
            ),

        "read_ok":
            False,

        "rows":
            np.nan,

        "missing_columns":
            None,

        "extra_columns":
            None,

        "duplicate_instance_id":
            np.nan,

        "ratio_violations":
            0,

        "binary_violations":
            0,

        "negative_violations":
            0,

        "valid":
            False,

        "error":
            None
    }

    try:

        df = pd.read_csv(
            path
        )

    except Exception as e:

        result[
            "error"
        ] = str(e)

        return result

    result[
        "read_ok"
    ] = True

    result[
        "rows"
    ] = len(
        df
    )

    missing = sorted(
        set(
            REQUIRED_EXPORT_COLUMNS
        )
        -
        set(
            df.columns
        )
    )

    extra = sorted(
        set(
            df.columns
        )
        -
        set(
            REQUIRED_EXPORT_COLUMNS
        )
    )

    result[
        "missing_columns"
    ] = " | ".join(
        missing
    )

    result[
        "extra_columns"
    ] = " | ".join(
        extra
    )

    if (
        "instance_id"
        in df.columns
    ):

        result[
            "duplicate_instance_id"
        ] = int(
            df[
                "instance_id"
            ]
            .duplicated()
            .sum()
        )

    ratio_violations = 0

    for col in RATIO_COLUMNS:

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        ratio_violations += int(
            (
                (
                    values
                    <
                    0
                )
                |
                (
                    values
                    >
                    1
                )
            ).sum()
        )

    binary_violations = 0

    for col in BINARY_COLUMNS:

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        binary_violations += int(
            (
                values
                .notna()
                &
                ~values.isin(
                    [
                        0,
                        1
                    ]
                )
            ).sum()
        )

    negative_violations = 0

    for col in (
        COUNT_COLUMNS
        +
        NONNEGATIVE_COLUMNS
    ):

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        negative_violations += int(
            (
                values
                <
                0
            ).sum()
        )

    result[
        "ratio_violations"
    ] = ratio_violations

    result[
        "binary_violations"
    ] = binary_violations

    result[
        "negative_violations"
    ] = negative_violations

    result[
        "valid"
    ] = (
        result[
            "read_ok"
        ]
        and
        len(
            missing
        )
        ==
        0
        and
        (
            result[
                "duplicate_instance_id"
            ]
            in [
                0,
                np.nan
            ]
            or
            pd.isna(
                result[
                    "duplicate_instance_id"
                ]
            )
        )
        and
        ratio_violations
        ==
        0
        and
        binary_violations
        ==
        0
        and
        negative_violations
        ==
        0
    )

    return result


## 12. Auditar exportaciones ya disponibles

In [31]:
export_files = sorted(
    EXPORT_DIR.glob(
        "*.csv"
    )
)

print(
    "Exports encontrados:",
    len(
        export_files
    )
)

export_audit = pd.DataFrame(
    [
        validate_extension_export(
            path
        )
        for path
        in export_files
    ]
)

if export_audit.empty:

    export_audit = pd.DataFrame(
        columns=[
            "file",
            "read_ok",
            "rows",
            "missing_columns",
            "extra_columns",
            "duplicate_instance_id",
            "ratio_violations",
            "binary_violations",
            "negative_violations",
            "valid",
            "error"
        ]
    )

display(
    export_audit
)


Exports encontrados: 0


,file,read_ok,rows,missing_columns,extra_columns,duplicate_instance_id,ratio_violations,binary_violations,negative_violations,valid,error


## 13. Auditoría de cobertura de exports

Cada instancia del manifiesto debe terminar con exactamente una exportación válida de la extensión.


In [32]:
if not pilot_manifest.empty:

    expected_export_names = set(
        (
            pilot_manifest[
                "instance_id"
            ]
            +
            ".csv"
        )
        .tolist()
    )

    observed_export_names = set(
        path.name
        for path
        in export_files
    )

    missing_exports = sorted(
        expected_export_names
        -
        observed_export_names
    )

    unexpected_exports = sorted(
        observed_export_names
        -
        expected_export_names
    )

else:

    missing_exports = []
    unexpected_exports = sorted(
        path.name
        for path
        in export_files
    )

export_coverage = pd.DataFrame([
    {
        "expected_instances":
            len(
                pilot_manifest
            ),

        "observed_exports":
            len(
                export_files
            ),

        "missing_export_count":
            len(
                missing_exports
            ),

        "unexpected_export_count":
            len(
                unexpected_exports
            ),

        "missing_exports":
            " | ".join(
                missing_exports
            ),

        "unexpected_exports":
            " | ".join(
                unexpected_exports
            )
    }
])

display(
    export_coverage
)


,expected_instances,observed_exports,missing_export_count,unexpected_export_count,missing_exports,unexpected_exports
0,48,0,48,0,site_001__task_01__exec_01.csv | site_001__task_01__exec_02.csv | site_001__task_02__exec_01.csv | site_001__task_02__exec_02.csv | site_002__task_01__exec_01.csv | site_002__task_01__exec_02.csv | site_002__task_02__exec_01.csv | site_002__task_02__exec_02.csv | site_003__task_01__exec_01.csv | site_003__task_01__exec_02.csv | site_003__task_02__exec_01.csv | site_003__task_02__exec_02.csv | ...,


## 14. Crear estructura de evidencia por instancia

In [33]:
evidence_structure_records = []

if not pilot_manifest.empty:

    for _, row in (
        pilot_manifest.iterrows()
    ):

        instance_dir = (
            EVIDENCE_DIR
            /
            row[
                "instance_id"
            ]
        )

        instance_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        expected_artifacts = [
            "task_context.json",
            "screenshot_before.png",
            "screenshot_after.png",
            "dom_snapshot_before.json",
            "dom_snapshot_after.json",
            "interaction_trace.json"
        ]

        evidence_structure_records.append({
            "instance_id":
                row[
                    "instance_id"
                ],

            "evidence_dir":
                str(
                    instance_dir.relative_to(
                        PROJECT_DIR
                    )
                ),

            "expected_artifacts":
                " | ".join(
                    expected_artifacts
                )
        })

evidence_structure = pd.DataFrame(
    evidence_structure_records
)

display(
    evidence_structure.head(
        20
    )
)


,instance_id,evidence_dir,expected_artifacts
0,site_001__task_01__exec_01,data/silver/raw/evidence/site_001__task_01__exec_01,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
1,site_001__task_01__exec_02,data/silver/raw/evidence/site_001__task_01__exec_02,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
2,site_001__task_02__exec_01,data/silver/raw/evidence/site_001__task_02__exec_01,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
3,site_001__task_02__exec_02,data/silver/raw/evidence/site_001__task_02__exec_02,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
4,site_002__task_01__exec_01,data/silver/raw/evidence/site_002__task_01__exec_01,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
5,site_002__task_01__exec_02,data/silver/raw/evidence/site_002__task_01__exec_02,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
6,site_002__task_02__exec_01,data/silver/raw/evidence/site_002__task_02__exec_01,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
7,site_002__task_02__exec_02,data/silver/raw/evidence/site_002__task_02__exec_02,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
8,site_003__task_01__exec_01,data/silver/raw/evidence/site_003__task_01__exec_01,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json
9,site_003__task_01__exec_02,data/silver/raw/evidence/site_003__task_01__exec_02,task_context.json | screenshot_before.png | screenshot_after.png | dom_snapshot_before.json | dom_snapshot_after.json | interaction_trace.json


Los snapshots/trace pueden quedar ausentes si la extensión aún no los produce, pero esa ausencia debe registrarse; no se inventa un artefacto vacío para simular observación.


## 15. Crear paquetes de anotación

In [34]:
annotation_package_records = []

if not pilot_manifest.empty:

    task_lookup = (
        included_tasks
        .set_index(
            "task_id"
        )
        .to_dict(
            orient="index"
        )
    )

    for _, row in (
        pilot_manifest.iterrows()
    ):

        instance_id = row[
            "instance_id"
        ]

        task_info = task_lookup.get(
            row[
                "task_id"
            ],
            {}
        )

        package = {
            "instance_id":
                instance_id,

            "site_id":
                row[
                    "site_id"
                ],

            "task_id":
                row[
                    "task_id"
                ],

            "execution_id":
                row[
                    "execution_id"
                ],

            "task_description":
                task_info.get(
                    "task_description"
                ),

            "start_condition":
                task_info.get(
                    "start_condition"
                ),

            "success_criterion":
                task_info.get(
                    "success_criterion"
                ),

            "applicable_H1":
                task_info.get(
                    "applicable_H1"
                ),

            "applicable_H4":
                task_info.get(
                    "applicable_H4"
                ),

            "applicable_H5":
                task_info.get(
                    "applicable_H5"
                ),

            "applicable_H6":
                task_info.get(
                    "applicable_H6"
                ),

            "evidence_dir":
                row[
                    "evidence_dir"
                ],

            "x_values_included":
                False,

            "model_prediction_included":
                False,

            "annotation_runs_required":
                3,

            "prompt_version":
                "silver_v1"
        }

        package_path = (
            ANNOTATION_PACKAGE_DIR
            /
            f"{instance_id}.json"
        )

        package_path.write_text(
            json.dumps(
                package,
                ensure_ascii=False,
                indent=2,
                default=str
            ),
            encoding="utf-8"
        )

        annotation_package_records.append({
            "instance_id":
                instance_id,

            "package_path":
                str(
                    package_path.relative_to(
                        PROJECT_DIR
                    )
                ),

            "created":
                package_path.exists()
        })

annotation_packages = pd.DataFrame(
    annotation_package_records
)

display(
    annotation_packages.head(
        20
    )
)


,instance_id,package_path,created
0,site_001__task_01__exec_01,data/silver/interim/annotation_packages/site_001__task_01__exec_01.json,True
1,site_001__task_01__exec_02,data/silver/interim/annotation_packages/site_001__task_01__exec_02.json,True
2,site_001__task_02__exec_01,data/silver/interim/annotation_packages/site_001__task_02__exec_01.json,True
3,site_001__task_02__exec_02,data/silver/interim/annotation_packages/site_001__task_02__exec_02.json,True
4,site_002__task_01__exec_01,data/silver/interim/annotation_packages/site_002__task_01__exec_01.json,True
5,site_002__task_01__exec_02,data/silver/interim/annotation_packages/site_002__task_01__exec_02.json,True
6,site_002__task_02__exec_01,data/silver/interim/annotation_packages/site_002__task_02__exec_01.json,True
7,site_002__task_02__exec_02,data/silver/interim/annotation_packages/site_002__task_02__exec_02.json,True
8,site_003__task_01__exec_01,data/silver/interim/annotation_packages/site_003__task_01__exec_01.json,True
9,site_003__task_01__exec_02,data/silver/interim/annotation_packages/site_003__task_01__exec_02.json,True


## 16. Checklist de recolección

In [35]:
collection_checklist = pd.DataFrame([
    {
        "step":
            1,

        "action":
            "Completar site_registry_pilot.csv",

        "done_when":
            "12 sitios incluidos, reproducibles y sin datos personales reales."
    },

    {
        "step":
            2,

        "action":
            "Completar task_registry_pilot.csv",

        "done_when":
            "24 tareas incluidas con inicio y criterio de éxito."
    },

    {
        "step":
            3,

        "action":
            "Reejecutar notebook 25",

        "done_when":
            "Registries ready = True y pilot_manifest = 48 filas."
    },

    {
        "step":
            4,

        "action":
            "Ejecutar cada tarea con la extensión",

        "done_when":
            "Existe un CSV por instance_id en extension_exports/."
    },

    {
        "step":
            5,

        "action":
            "Guardar evidencia bruta",

        "done_when":
            "Capturas/snapshots/trace quedan asociados al mismo instance_id."
    },

    {
        "step":
            6,

        "action":
            "Reejecutar notebook 25",

        "done_when":
            "48 exports presentes y válidos."
    },

    {
        "step":
            7,

        "action":
            "Ejecutar 3 anotaciones LLM por instancia",

        "done_when":
            "144 label runs con procedencia registrada."
    },

    {
        "step":
            8,

        "action":
            "Construir consenso",

        "done_when":
            "Targets 0/1/NaN por instancia sin rellenar NaN con 0."
    }
])

display(
    collection_checklist
)


,step,action,done_when
0,1,Completar site_registry_pilot.csv,"12 sitios incluidos, reproducibles y sin datos personales reales."
1,2,Completar task_registry_pilot.csv,24 tareas incluidas con inicio y criterio de éxito.
2,3,Reejecutar notebook 25,Registries ready = True y pilot_manifest = 48 filas.
3,4,Ejecutar cada tarea con la extensión,Existe un CSV por instance_id en extension_exports/.
4,5,Guardar evidencia bruta,Capturas/snapshots/trace quedan asociados al mismo instance_id.
5,6,Reejecutar notebook 25,48 exports presentes y válidos.
6,7,Ejecutar 3 anotaciones LLM por instancia,144 label runs con procedencia registrada.
7,8,Construir consenso,Targets 0/1/NaN por instancia sin rellenar NaN con 0.


## 17. Estado general del preflight

In [36]:
registries_pass = bool(
    registries_ready
)

manifest_pass = (
    len(
        pilot_manifest
    )
    ==
    EXPECTED_INSTANCES
)

exports_valid_count = (
    int(
        export_audit[
            "valid"
        ].fillna(
            False
        ).sum()
    )
    if not export_audit.empty
    else 0
)

exports_pass = (
    manifest_pass
    and
    len(
        export_files
    )
    ==
    EXPECTED_INSTANCES
    and
    exports_valid_count
    ==
    EXPECTED_INSTANCES
    and
    len(
        missing_exports
    )
    ==
    0
)

if not registries_pass:

    pilot_status = (
        "WAITING_FOR_SITE_TASK_REGISTRY"
    )

elif not manifest_pass:

    pilot_status = (
        "MANIFEST_NOT_READY"
    )

elif not exports_pass:

    pilot_status = (
        "READY_FOR_COLLECTION"
    )

else:

    pilot_status = (
        "EXTENSION_CAPTURE_COMPLETE"
    )

preflight_status = pd.DataFrame([
    {
        "registries_ready":
            registries_pass,

        "manifest_rows":
            len(
                pilot_manifest
            ),

        "expected_manifest_rows":
            EXPECTED_INSTANCES,

        "exports_found":
            len(
                export_files
            ),

        "valid_exports":
            exports_valid_count,

        "pilot_status":
            pilot_status
    }
])

display(
    preflight_status
)


,registries_ready,manifest_rows,expected_manifest_rows,exports_found,valid_exports,pilot_status
0,True,48,48,0,0,READY_FOR_COLLECTION


## 18. Guardar auditorías

In [37]:
site_validation.to_csv(
    REPORTS_DIR
    / "silver_pilot_site_validation.csv",
    index=False
)

task_validation.to_csv(
    REPORTS_DIR
    / "silver_pilot_task_validation.csv",
    index=False
)

export_audit.to_csv(
    REPORTS_DIR
    / "silver_pilot_export_audit.csv",
    index=False
)

export_coverage.to_csv(
    REPORTS_DIR
    / "silver_pilot_export_coverage.csv",
    index=False
)

evidence_structure.to_csv(
    REPORTS_DIR
    / "silver_pilot_evidence_structure.csv",
    index=False
)

annotation_packages.to_csv(
    REPORTS_DIR
    / "silver_pilot_annotation_packages.csv",
    index=False
)

collection_checklist.to_csv(
    REPORTS_DIR
    / "silver_pilot_collection_checklist.csv",
    index=False
)

preflight_status.to_csv(
    REPORTS_DIR
    / "silver_pilot_preflight_status.csv",
    index=False
)


In [38]:
expected_outputs = [
    META_DIR
    / "site_registry_pilot.csv",

    META_DIR
    / "task_registry_pilot.csv",

    META_DIR
    / "pilot_manifest.csv",

    REPORTS_DIR
    / "silver_pilot_site_validation.csv",

    REPORTS_DIR
    / "silver_pilot_task_validation.csv",

    REPORTS_DIR
    / "silver_pilot_export_audit.csv",

    REPORTS_DIR
    / "silver_pilot_export_coverage.csv",

    REPORTS_DIR
    / "silver_pilot_evidence_structure.csv",

    REPORTS_DIR
    / "silver_pilot_annotation_packages.csv",

    REPORTS_DIR
    / "silver_pilot_collection_checklist.csv",

    REPORTS_DIR
    / "silver_pilot_preflight_status.csv"
]

for path in expected_outputs:

    print(
        path.relative_to(
            PROJECT_DIR
        ),
        "✅"
        if path.exists()
        else "❌"
    )


data/silver/metadata/site_registry_pilot.csv ✅
data/silver/metadata/task_registry_pilot.csv ✅
data/silver/metadata/pilot_manifest.csv ✅
reports/tables/silver_pilot_site_validation.csv ✅
reports/tables/silver_pilot_task_validation.csv ✅
reports/tables/silver_pilot_export_audit.csv ✅
reports/tables/silver_pilot_export_coverage.csv ✅
reports/tables/silver_pilot_evidence_structure.csv ✅
reports/tables/silver_pilot_annotation_packages.csv ✅
reports/tables/silver_pilot_collection_checklist.csv ✅
reports/tables/silver_pilot_preflight_status.csv ✅


## Punto de detención

En la primera ejecución es normal obtener:

```text
pilot_status = WAITING_FOR_SITE_TASK_REGISTRY
```

porque las plantillas se crean vacías intencionalmente.

El siguiente trabajo ya no es otra auditoría de datasets públicos. Deben completarse:

- `data/silver/metadata/site_registry_pilot.csv`
- `data/silver/metadata/task_registry_pilot.csv`

Después se reejecuta este mismo notebook.

Cuando el resultado sea:

```text
registries_ready = True
manifest_rows = 48
pilot_status = READY_FOR_COLLECTION
```

se puede iniciar la captura real con la extensión.
